In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.io import load_dataset, load_fine


DATA_DIR = REPO_ROOT / "DATA"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

fine = load_fine(
    DATA_DIR,
    "fine.csv",
)

print(f"Repository root : {REPO_ROOT}")
print(f"Data directory  : {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Load the aggregated mid-level data

aggregated_mid = pd.read_csv(DATA_DIR/"aggregated_mid.csv",sep=";",index_col=0)

aggregated_mid = aggregated_mid.drop(
    index="PM",
    errors="ignore",
)

pm_regions = (
    fine.loc[
        fine.index.get_level_values("mid").isin(["PMd", "PMv"])
    ]
    .groupby(level="mid")
    .mean()
    .round(2)
)

aggregated_mid = pd.concat(
    [
        aggregated_mid,
        pm_regions,
    ]
)

In [ ]:
# Calculate group-level fold changes relative to OPCRT and retain only regions
# identified as significant by the PLS analysis

grouped_columns = {}

for column in aggregated_mid.columns:
    group = "".join(char for char in column if not char.isdigit())
    grouped_columns.setdefault(group, []).append(column)

group_means = pd.DataFrame(
    {
        group: aggregated_mid[columns].mean(axis=1)
        for group, columns in grouped_columns.items()
    }
)

fold_change = pd.DataFrame(
    {
        "HC": group_means["OPCRT"] / group_means["HC"],
        "CNTX": group_means["OPCRT"] / group_means["CNTX"],
        "OCT": group_means["OPCRT"] / group_means["OCT"],
    }
)

significant_regions = pd.read_csv(
    DATA_DIR / "pls_significant_regions.csv",
    index_col=0,
)


significant_fold_change = {
    "HC": fold_change[["HC"]].loc[
        fold_change.index.intersection(
            significant_regions["HC_OPCRT"].dropna().index
        )
    ],
    "CNTX": fold_change[["CNTX"]].loc[
        fold_change.index.intersection(
            significant_regions["CNTX_OPCRT"].dropna().index
        )
    ],
    "OCT": fold_change[["OCT"]].loc[
        fold_change.index.intersection(
            significant_regions["OCT_OPCRT"].dropna().index
        )
    ],
}

In [ ]:
# Map significant fold changes onto brain atlas sections at selected depths

from braian import BrainData
import braian.plot as bap

brain_regions = aggregated_mid.index.tolist()
depths = (2500, 3500, 5000, 7000, 8000, 9000)

for comparison, data in significant_fold_change.items():

    comparison_output = OUTPUT_DIR / comparison
    comparison_output.mkdir(exist_ok=True)

    brain_data = BrainData(
        data=data[comparison],
        name=comparison,
        metric="fold change",
        units="fold change",
    )

    bap.heatmap(
        brain_data,
        brain_regions,
        orientation="frontal",
        depth=depths,
        cmap="Blues",
        cmin=0,
        cmax=float(data.max()),
        show_acronyms=False,
        output_path=str(comparison_output),
        filename=f"{comparison}_frontal",
    )

    print(f"Saved {comparison} heatmap to: {comparison_output}")

In [ ]:
# Compare the significant regions identified in the three group comparisons 
# and display their number in the Venn diagram

max_regions = max(
    len(significant_fold_change["HC"]),
    len(significant_fold_change["CNTX"]),
    len(significant_fold_change["OCT"]),
)

pls_results_aggregated = pd.DataFrame(
    {
        comparison: (
            significant_fold_change[comparison]
            .index.to_list()
            + [np.nan] * (max_regions - len(significant_fold_change[comparison]))
        )
        for comparison in ["HC", "CNTX", "OCT"]
    }
)

import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from matplotlib.colors import to_rgb

set_hc = set(significant_fold_change["HC"].index)
set_cntx = set(significant_fold_change["CNTX"].index)
set_oct = set(significant_fold_change["OCT"].index)

fig, ax = plt.subplots(
    figsize=(10, 10),
)

venn = venn3(
    [set_hc, set_cntx, set_oct],
    set_labels=("HC", "CNTX", "OCT"),
    ax=ax,
)

for label in venn.set_labels:
    if label:
        label.set_fontsize(22)
        label.set_fontweight("bold")
        label.set_color("#222222")

for label in venn.subset_labels:
    if label:
        label.set_fontsize(18)
        label.set_fontweight("bold")
        label.set_color("#111111")


colors = {
    "100": "#d8ecd2",
    "010": "#A8DADC",
    "001": "#457B9D",
}


def blend_colors(color_1, color_2):
    rgb_1 = to_rgb(color_1)
    rgb_2 = to_rgb(color_2)

    return tuple(
        (value_1 + value_2) / 2
        for value_1, value_2 in zip(rgb_1, rgb_2)
    )


for subset_id in ["100", "010", "001"]:
    patch = venn.get_patch_by_id(subset_id)

    if patch:
        patch.set_color(colors[subset_id])
        patch.set_alpha(0.7)


double_intersections = {
    "110": blend_colors(colors["100"], colors["010"]),
    "101": blend_colors(colors["100"], colors["001"]),
    "011": blend_colors(colors["010"], colors["001"]),
}

for subset_id, color in double_intersections.items():
    patch = venn.get_patch_by_id(subset_id)

    if patch:
        patch.set_color(color)
        patch.set_alpha(0.8)


triple_color = tuple(
    sum(rgb_values) / 3
    for rgb_values in zip(
        to_rgb(colors["100"]),
        to_rgb(colors["010"]),
        to_rgb(colors["001"]),
    )
)

patch = venn.get_patch_by_id("111")

if patch:
    patch.set_color(triple_color)
    patch.set_alpha(1.0)


plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "Venn.svg",
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Download the Allen Institute connectivity dataset if needed, then aggregate
# connectivity values by target region

from io import StringIO
import requests


FIGURE_DIR = REPO_ROOT / "Fig_3"
conn_file = FIGURE_DIR / "allen_connectivity_fine.csv"

allen_url = (
    "http://download.alleninstitute.org/publications/"
    "A_high_resolution_data-driven_model_of_the_mouse_connectome/"
    "normalized_connection_density.csv"
)


if not conn_file.exists():

    response = requests.get(allen_url)
    response.raise_for_status()

    conn_file.write_text(
        response.text,
        encoding="utf-8",
    )

    print(f"Downloaded: {conn_file}")

else:
    print(f"Using existing file: {conn_file}")


conn = pd.read_csv(
    conn_file,
    sep=",",
    header=[0, 1],
    index_col=0,
)


conn_merged = (
    conn.groupby(
        level=1,
        axis=1,
        sort=False,
    )
    .mean()
)

conn_merged

In [ ]:
# Test whether connectivity among the selected significant regions is higher than
# expected by chance using a permutation-based null distribution

significant_regions = ['ACA', 'PL', 'ILA', 'ORB', 'RSP', 'POST']

observed_matrix = conn_merged.loc[
    significant_regions,
    significant_regions,
].values

observed_values = observed_matrix[
    np.triu_indices_from(observed_matrix, k=1)
]

observed_mean_connectivity = np.nanmean(
    observed_values
)

n_permutations = 10000
random_state = 42

rng = np.random.default_rng(random_state)

all_regions = conn_merged.index.to_list()
n_selected_regions = len(significant_regions)

permutation_means = np.zeros(
    n_permutations
)


for permutation in range(n_permutations):

    random_regions = rng.choice(
        all_regions,
        size=n_selected_regions,
        replace=False,
    )

    random_matrix = conn_merged.loc[
        random_regions,
        random_regions,
    ].values

    random_values = random_matrix[
        np.triu_indices_from(random_matrix, k=1)
    ]

    permutation_means[permutation] = np.nanmean(
        random_values
    )


p_value = np.mean(
    permutation_means >= observed_mean_connectivity
)

print(
    f"Observed mean connectivity: {observed_mean_connectivity:.5f}"
)

print(
    f"Permutation mean connectivity: {permutation_means.mean():.5f}"
)

print(
    f"Permutation p-value: {p_value:.4f}"
)

In [ ]:
# Plot the permutation distribution of mean connectivity and indicate the observed value

fig, ax = plt.subplots(
    figsize=(9, 6),
)

sns.histplot(
    permutation_means,
    bins=50,
    color="#60bfc8",
    alpha=0.7,
    stat="count",
    kde=True,
    ax=ax,
)

ax.axvline(
    observed_mean_connectivity,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Observed",
)

ax.set_xlabel(
    "Mean Connectivity",
    fontsize=14,
)

ax.set_ylabel(
    "Frequency",
    fontsize=14,
)

ax.set_xlim(
    0,
    0.0004,
)

ax.grid(False)

ax.legend(
    fontsize=13,
)

ax.tick_params(
    axis="both",
    labelsize=12,
)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "connectivity_permutation_distribution.svg",
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Visualize the anatomical connectivity network of significant regions
import networkx as nx
import igraph as ig
from utils.plot import plot_connectivity_network

mid_conn = conn_merged.loc[significant_regions, significant_regions].copy()
mid_conn = mid_conn.dropna(how="all", axis=0).dropna(how="all", axis=1)
np.fill_diagonal(mid_conn.values, 0)

graph = nx.DiGraph()
for source in mid_conn.index:
    for target in mid_conn.columns:
        if source != target:
            weight = mid_conn.loc[source, target]
            if pd.notna(weight):
                graph.add_edge(source, target, weight=weight)

fig, ax = plot_connectivity_network(graph)

plt.tight_layout()
fig.savefig(
    OUTPUT_DIR / "connectivity_network_significant_regions.svg",
    bbox_inches="tight",
)
plt.show()